# Triana et al. (2021) Healthy Bone-Marrow Reference Projection

Projects single-cell data onto the Triana et al. (2021) healthy bone-marrow reference and exports projected cell-type labels as a CSV.

**Reference:**  
Triana S et al. *Single-cell proteo-genomic reference maps of the hematopoietic system enable the purification and massive profiling of precisely defined cell states.* Nature Immunology (2021).  
https://doi.org/10.1038/s41590-021-01059-0

**Reference object (figshare):**  
https://doi.org/10.6084/m9.figshare.13397651

---

### ⚠️ Before you start
1. Go to **Runtime → Change runtime type** and set it to **R**.
2. Upload your query `.h5ad` file using the cell below, or mount Google Drive.
3. Set your parameters in the **Parameters** cell.
4. Run all cells top-to-bottom (**Runtime → Run all**).

## 1 · Install packages

This takes **~10–15 minutes** on a fresh Colab runtime. Subsequent runs reuse the cached packages and are much faster.

In [ ]:
# Install BiocManager if needed, then all required packages.
# Re-running this cell on an already-configured runtime is safe (skips installed packages).

if (!requireNamespace("BiocManager", quietly = TRUE)) {
  install.packages("BiocManager", repos = "https://cloud.r-project.org")
}

bioc_pkgs <- c("SingleCellExperiment", "scmap", "zellkonverter", "BiocParallel")
cran_pkgs <- c("Seurat", "plyr")

missing_bioc <- bioc_pkgs[!sapply(bioc_pkgs, requireNamespace, quietly = TRUE)]
missing_cran <- cran_pkgs[!sapply(cran_pkgs, requireNamespace, quietly = TRUE)]

if (length(missing_bioc) > 0) {
  message("Installing Bioconductor packages: ", paste(missing_bioc, collapse = ", "))
  BiocManager::install(missing_bioc, ask = FALSE, update = FALSE)
}
if (length(missing_cran) > 0) {
  message("Installing CRAN packages: ", paste(missing_cran, collapse = ", "))
  install.packages(missing_cran, repos = "https://cloud.r-project.org")
}

message("✅ All packages ready.")

## 2 · Upload your query file

Run the cell below to open a file-picker and upload your `.h5ad` query file directly into the Colab session.  
Alternatively, mount Google Drive in the next cell and point `QUERY_H5AD_PATH` at a Drive path.

In [ ]:
# ── Option A: interactive upload ─────────────────────────────────────────────
# Colab's file-picker widget (only works in Colab; skip if using Drive).

# This uses Python via system2 to trigger the JS upload widget.
# After upload the file will land in /content/.

system2("python3", args = c("-c", shQuote("
from google.colab import files
uploaded = files.upload()
for fn in uploaded:
    print('Uploaded:', fn)
")))

# ── Option B: mount Google Drive ─────────────────────────────────────────────
# Uncomment and run instead of Option A.
#
# system2("python3", args = c("-c", shQuote("
# from google.colab import drive
# drive.mount('/content/drive')
# ")))
#
# Then set QUERY_H5AD_PATH below to something like:
#   "/content/drive/MyDrive/my_project/query.h5ad"

## 3 · Parameters

Edit these variables — they replace the command-line arguments of the original script.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║                     USER PARAMETERS                             ║
# ╚══════════════════════════════════════════════════════════════════╝

# Path to your query .h5ad file (uploaded above or on Drive)
QUERY_H5AD_PATH <- "velten.h5ad"   # ← CHANGE THIS

# Where to write the output CSV
OUTPUT_CSV_PATH <- "output_labels.csv"

# (Optional) Path to a locally cached Triana reference .rds.
# Leave as NULL to auto-download from figshare (~2 GB).
REFERENCE_RDS_PATH <- "triana_healthy_reference.rds"  # ← CHANGE THIS or set to NULL to download automatically

# Number of nearest neighbours for scmap projection
K_NEIGHBOURS <- 5L

# Random seed for reproducibility
SEED <- 42L

# ───────────────────────────────────────────────────────────────────
set.seed(SEED)
message("Parameters set:")
message("  Query  : ", QUERY_H5AD_PATH)
message("  Output : ", OUTPUT_CSV_PATH)
message("  Reference: ", ifelse(is.null(REFERENCE_RDS_PATH), "[auto-download]", REFERENCE_RDS_PATH))
message("  k      : ", K_NEIGHBOURS)
message("  seed   : ", SEED)

## 4 · Load libraries

In [ ]:
suppressPackageStartupMessages({
  library(Seurat)
  library(SingleCellExperiment)
  library(scmap)
  library(zellkonverter)
  library(BiocParallel)
  library(plyr)
})
message("✅ Libraries loaded.")

## 5 · Helper functions

In [ ]:
# ── Download Triana reference from figshare ───────────────────────────────────
#
# The dataset at https://doi.org/10.6084/m9.figshare.13397651 contains a
# Seurat object of annotated healthy bone-marrow cells (~2 GB download).
# The file is cached in /content/ so repeated runs within the same session
# skip the download.

download_triana_reference <- function(dest_dir = "/content") {
  dest_file <- file.path(dest_dir, "triana_healthy_reference.rds")

  if (file.exists(dest_file)) {
    message("Using cached reference: ", dest_file)
    return(dest_file)
  }

  message("Fetching figshare file list for dataset 13397651 ...")
  api_url <- "https://api.figshare.com/v2/articles/13397651/files"

  resp <- tryCatch(
    readLines(url(api_url), warn = FALSE),
    error = function(e) stop("Cannot reach figshare API: ", conditionMessage(e))
  )
  close(url(api_url))
  json <- paste(resp, collapse = "")

  # Extract .rds download URL
  rds_urls <- regmatches(json, gregexpr('"download_url":"[^"]+\.rds"', json))[[1]]
  if (length(rds_urls) == 0)
    rds_urls <- regmatches(json, gregexpr('"download_url":"[^"]+\.RDS"', json))[[1]]

  if (length(rds_urls) == 0) {
    stop(
      "No .rds file found in figshare dataset 13397651.\n",
      "Please manually download the reference from:\n",
      "  https://doi.org/10.6084/m9.figshare.13397651\n",
      "then set REFERENCE_RDS_PATH to its local path."
    )
  }

  dl_url <- sub('"download_url":"', "", rds_urls[[1]])
  dl_url <- sub('"$', "", dl_url)

  message("Downloading Triana reference (~2 GB) from:\n  ", dl_url)
  message("Destination: ", dest_file)
  message("This may take several minutes ...")

  download.file(dl_url, dest_file, mode = "wb", quiet = FALSE)
  message("✅ Download complete.")
  dest_file
}

In [ ]:
# ── scmap projection onto Triana reference ────────────────────────────────────
#
# Adapted from `project_anyref()` in the Triana Projection_Vignette:
# https://git.embl.org/triana/nrn/-/tree/master/Projection_Vignette
#
# @param query_sce   SingleCellExperiment with log-normalised counts (logcounts)
# @param ref_seurat  Seurat reference with a 'celltype' metadata column
# @param k           Number of nearest neighbours
# @param features    Gene names to use; NULL → reference variable features
# @return data.frame: barcode | projected_celltype | correlation_score | pseudotime

project_to_reference <- function(query_sce, ref_seurat, k = 5L, features = NULL) {

  # 1. Determine projection genes ──────────────────────────────────────────────
  if (is.null(features)) {
    features <- VariableFeatures(ref_seurat)
    if (length(features) == 0)
      stop("Reference has no variable features. Run FindVariableFeatures() first.")
    message("  Using ", length(features), " variable features from reference.")
  }

  common_genes <- Reduce(intersect, list(features, rownames(query_sce), rownames(ref_seurat)))
  if (length(common_genes) < 50)
    stop("Fewer than 50 genes shared between query and reference (",
         length(common_genes), " found). Check that gene name conventions match.")
  message("  Projecting using ", length(common_genes), " common genes.")

  query_sub <- query_sce[common_genes, ]

  # 2. Build reference SCE ─────────────────────────────────────────────────────
  ref_mat <- GetAssayData(ref_seurat, slot = "data")   # log-normalised
  ref_sce <- SingleCellExperiment(
    assays  = list(normcounts = as.matrix(ref_mat[common_genes, ])),
    colData = ref_seurat@meta.data
  )
  logcounts(ref_sce) <- assay(ref_sce, "normcounts")

  # 3. Identify metadata columns ───────────────────────────────────────────────
  ct_candidates <- c("celltype", "cell_type", "CellType", "Annotation",
                     "annotation", "label", "cluster")
  ct_col <- intersect(ct_candidates, colnames(colData(ref_sce)))
  if (length(ct_col) == 0)
    stop("No cell-type column found in reference metadata.\n",
         "Available columns: ", paste(colnames(colData(ref_sce)), collapse = ", "))
  ct_col <- ct_col[1]
  message("  Cell-type column: '", ct_col, "'")

  pt_candidates <- c("Myelocytes", "pseudotime", "Pseudotime", "dpt_pseudotime")
  pt_col        <- intersect(pt_candidates, colnames(colData(ref_sce)))
  has_pt        <- length(pt_col) > 0
  if (has_pt) pt_col <- pt_col[1]

  # 4. Build scmap cell index on reference ─────────────────────────────────────
  rowData(ref_sce)$feature_symbol   <- rownames(ref_sce)
  rowData(query_sub)$feature_symbol <- rownames(query_sub)

  ref_sce <- selectFeatures(ref_sce, suppress_plot = TRUE)
  ref_sce <- indexCell(ref_sce)

  # 5. kNN projection ──────────────────────────────────────────────────────────
  message("  Running scmap kNN projection (k=", k, ") ...")
  proj <- scmapCell(
    projection = query_sub,
    index_list = list(ref = metadata(ref_sce)$scmap_cell_index),
    w          = k
  )

  nn_idx    <- proj$ref$cells        # k × n_query
  nn_scores <- proj$ref$similarities # k × n_query
  n_query   <- ncol(query_sub)

  ref_labels <- as.character(colData(ref_sce)[[ct_col]])
  if (has_pt) ref_pt <- as.numeric(colData(ref_sce)[[pt_col]])

  proj_types <- character(n_query)
  proj_score <- numeric(n_query)
  proj_pt    <- rep(NA_real_, n_query)

  for (i in seq_len(n_query)) {
    nn    <- nn_idx[, i]
    valid <- !is.na(nn) & nn > 0
    if (!any(valid)) {
      proj_types[i] <- NA_character_
      proj_score[i] <- NA_real_
      next
    }
    nn_v          <- nn[valid]
    proj_types[i] <- names(sort(table(ref_labels[nn_v]), decreasing = TRUE))[1]
    proj_score[i] <- mean(nn_scores[valid, i], na.rm = TRUE)
    if (has_pt) proj_pt[i] <- mean(ref_pt[nn_v], na.rm = TRUE)
  }

  data.frame(
    barcode            = colnames(query_sub),
    projected_celltype = proj_types,
    correlation_score  = proj_score,
    pseudotime         = proj_pt,
    stringsAsFactors   = FALSE
  )
}

message("✅ Helper functions defined.")

## 6 · Run projection

In [ ]:
# ── Step 1: locate / download reference ──────────────────────────────────────
message("=== Triana Reference Projection ===")

if (!is.null(REFERENCE_RDS_PATH)) {
  if (!file.exists(REFERENCE_RDS_PATH))
    stop("Reference file not found: ", REFERENCE_RDS_PATH)
  ref_path <- REFERENCE_RDS_PATH
} else {
  ref_path <- download_triana_reference("/content")
}

message("\nLoading reference: ", ref_path)
ref_seurat <- readRDS(ref_path)
message("  Reference cells : ", ncol(ref_seurat))
message("  Reference genes : ", nrow(ref_seurat))

In [ ]:
# ── Step 2: load query .h5ad ─────────────────────────────────────────────────
if (!file.exists(QUERY_H5AD_PATH))
  stop("Query file not found: ", QUERY_H5AD_PATH,
       "\nDid you upload it in Section 2?")

message("Loading query: ", QUERY_H5AD_PATH)
query_sce <- readH5AD(QUERY_H5AD_PATH, use_hdf5 = FALSE)
message("  Query cells : ", ncol(query_sce))
message("  Query genes : ", nrow(query_sce))
message("  Assays      : ", paste(assayNames(query_sce), collapse = ", "))

# Ensure log-normalised counts are in the 'logcounts' slot
if (!"logcounts" %in% assayNames(query_sce)) {
  if ("X" %in% assayNames(query_sce)) {
    message("  Mapping assay 'X' → 'logcounts'")
    logcounts(query_sce) <- assay(query_sce, "X")
  } else {
    stop(
      "Cannot find log-normalised expression in the query .h5ad.\n",
      "Available assays: ", paste(assayNames(query_sce), collapse = ", "), "\n",
      "Please ensure the object was saved with log-normalised counts in .X or logcounts."
    )
  }
}

In [ ]:
# ── Step 3: run projection ───────────────────────────────────────────────────
message("\nRunning projection ...")

results <- project_to_reference(
  query_sce  = query_sce,
  ref_seurat = ref_seurat,
  k          = K_NEIGHBOURS
)

message("\nProjection complete. Preview:")
head(results)

In [ ]:
# ── Step 4: save output CSV ───────────────────────────────────────────────────
dir.create(dirname(OUTPUT_CSV_PATH), showWarnings = FALSE, recursive = TRUE)
write.csv(results, OUTPUT_CSV_PATH, row.names = FALSE, quote = TRUE)

message("\n✅ Done. ", nrow(results), " cells written to: ", OUTPUT_CSV_PATH)

n_na <- sum(is.na(results$projected_celltype))
if (n_na > 0)
  message("  ⚠️  Warning: ", n_na, " cells could not be projected (NA in output).")

message("\nLabel distribution:")
print(sort(table(results$projected_celltype), decreasing = TRUE))

## 7 · Download the output CSV

In [ ]:
# Trigger a browser download of the results CSV.
# This uses Python's Colab files helper from within R.

system2("python3", args = c("-c", shQuote(paste0("
from google.colab import files
files.download('", OUTPUT_CSV_PATH, "')
"))))

message("📥 Download triggered for: ", OUTPUT_CSV_PATH)

## 8 · Optional: quick visualisation

In [ ]:
# Bar chart of projected cell-type counts
ct_counts <- sort(table(results$projected_celltype), decreasing = TRUE)
par(mar = c(10, 4, 3, 1))
barplot(
  ct_counts,
  las    = 2,
  col    = colorRampPalette(c("#4E79A7", "#F28E2B", "#E15759", "#76B7B2"))(length(ct_counts)),
  main   = "Projected cell-type distribution",
  ylab   = "Number of cells",
  cex.names = 0.75
)

In [ ]:
# Histogram of scmap correlation scores
hist(
  results$correlation_score,
  breaks = 40,
  col    = "#4E79A7",
  border = "white",
  main   = "scmap correlation score distribution",
  xlab   = "Correlation score",
  ylab   = "Number of cells"
)
abline(v = 0.5, col = "firebrick", lty = 2, lwd = 1.5)
legend("topleft", legend = "score = 0.5", col = "firebrick", lty = 2, bty = "n")

In [ ]:
# Per-cell-type score box plot (shows projection confidence per label)
ct_order <- names(sort(tapply(results$correlation_score,
                               results$projected_celltype, median),
                        decreasing = TRUE))
par(mar = c(11, 4, 3, 1))
boxplot(
  correlation_score ~ factor(projected_celltype, levels = ct_order),
  data    = results,
  las     = 2,
  col     = "#76B7B2",
  outline = FALSE,
  main    = "Correlation score by projected cell type",
  ylab    = "Correlation score",
  xlab    = "",
  cex.axis = 0.75
)

## Notes

| Output column | Description |
|---|---|
| `barcode` | Cell barcode from the query object |
| `projected_celltype` | Majority-vote cell type from k nearest reference neighbours |
| `correlation_score` | Mean scmap cosine similarity (0–1; higher = better match) |
| `pseudotime` | Mean pseudotime of reference nearest neighbours (endpoint: Myelocytes); `NA` if no pseudotime column found |

**Tip:** cells with a `correlation_score` below ~0.5 are poorly matched and should be interpreted with caution.